# Notebook — Teste do módulo `collocation.py`

Este notebook testa o módulo:

```python
geodesy/collocation.py
```

Serão testadas as funções:

- `empirical_covariance`
- `covariance_model_gaussian`
- `covariance_model_exponential`
- `distance_matrix`
- `covariance_matrix`
- `least_squares_collocation`
- `collocation_error_variance`

Os exemplos simulam observações esparsas de um campo geodésico/geofísico e aplicam colocação por mínimos quadrados para predizer o campo em uma grade regular.


In [ ]:
# ============================================================
# 0. CONFIGURAÇÃO INICIAL
# ============================================================

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

current_dir = Path.cwd()

if (current_dir / "geodesy").exists():
    sys.path.insert(0, str(current_dir))
    print("Pasta geodesy encontrada no diretório atual.")
else:
    print("Atenção: a pasta geodesy não foi encontrada no diretório atual.")
    print("Coloque este notebook no mesmo diretório da pasta geodesy/.")

import geodesy
from geodesy import collocation

print("Versão do pacote geodesy:", geodesy.__version__)


## 1. Campo sintético verdadeiro


In [ ]:
# ============================================================
# 1. CAMPO SINTÉTICO VERDADEIRO EM GRADE
# ============================================================

lon = np.linspace(-55, -50, 90)
lat = np.linspace(-5, -1, 70)

LON, LAT = np.meshgrid(lon, lat)

FIELD_TRUE = (
    20.0 * np.exp(-((LON + 52.5)**2 + (LAT + 3.0)**2) / 0.35)
    - 12.0 * np.exp(-((LON + 53.6)**2 + (LAT + 2.0)**2) / 0.20)
    + 4.0 * np.sin((LON + 55.0) * 2*np.pi/5.0) * np.cos((LAT + 5.0) * np.pi/4.0)
)

plt.figure(figsize=(8, 5))
c = plt.contourf(LON, LAT, FIELD_TRUE, levels=30, cmap="coolwarm")
plt.colorbar(c, label="Campo sintético")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Campo verdadeiro sintético")
plt.grid(alpha=0.2)
plt.show()

print("Campo verdadeiro min/max:", np.nanmin(FIELD_TRUE), np.nanmax(FIELD_TRUE))


## 2. Amostragem esparsa do campo


In [ ]:
# ============================================================
# 2. AMOSTRAGEM ESPARSA
# ============================================================

rng = np.random.default_rng(42)

n_obs = 120

lon_obs = rng.uniform(lon.min(), lon.max(), n_obs)
lat_obs = rng.uniform(lat.min(), lat.max(), n_obs)

# Campo verdadeiro nos pontos observados usando a mesma função sintética
values_true_obs = (
    20.0 * np.exp(-((lon_obs + 52.5)**2 + (lat_obs + 3.0)**2) / 0.35)
    - 12.0 * np.exp(-((lon_obs + 53.6)**2 + (lat_obs + 2.0)**2) / 0.20)
    + 4.0 * np.sin((lon_obs + 55.0) * 2*np.pi/5.0) * np.cos((lat_obs + 5.0) * np.pi/4.0)
)

noise = rng.normal(0.0, 1.0, n_obs)
values_obs = values_true_obs + noise

plt.figure(figsize=(8, 5))
c = plt.contourf(LON, LAT, FIELD_TRUE, levels=30, cmap="coolwarm")
plt.colorbar(c, label="Campo verdadeiro")
plt.scatter(lon_obs, lat_obs, c=values_obs, edgecolor="k")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Observações esparsas sobre o campo verdadeiro")
plt.grid(alpha=0.2)
plt.show()

print("Observações min/max:", values_obs.min(), values_obs.max())


## 3. Matriz de distâncias


In [ ]:
# ============================================================
# 3. MATRIZ DE DISTÂNCIAS
# ============================================================

D = collocation.distance_matrix(lon_obs, lat_obs)

print("Shape da matriz de distâncias:", D.shape)
print("Distância mínima:", np.min(D))
print("Distância máxima:", np.max(D))

plt.figure(figsize=(6, 5))
plt.imshow(D)
plt.colorbar(label="Distância em graus")
plt.title("Matriz de distâncias entre observações")
plt.xlabel("Observação")
plt.ylabel("Observação")
plt.show()


## 4. Modelos de covariância


In [ ]:
# ============================================================
# 4. MODELOS DE COVARIÂNCIA
# ============================================================

dist = np.linspace(0, 5, 300)

sigma2 = np.var(values_obs)
corr_lengths = [0.2, 0.5, 1.0, 2.0]

plt.figure(figsize=(10, 4))

for Lcorr in corr_lengths:
    Cg = collocation.covariance_model_gaussian(dist, sigma2=sigma2, correlation_length=Lcorr)
    plt.plot(dist, Cg, label=f"Gauss L={Lcorr}")

plt.xlabel("Distância")
plt.ylabel("Covariância")
plt.title("Modelo gaussiano de covariância")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))

for Lcorr in corr_lengths:
    Ce = collocation.covariance_model_exponential(dist, sigma2=sigma2, correlation_length=Lcorr)
    plt.plot(dist, Ce, label=f"Exp L={Lcorr}")

plt.xlabel("Distância")
plt.ylabel("Covariância")
plt.title("Modelo exponencial de covariância")
plt.legend()
plt.grid(True)
plt.show()


## 5. Matriz de covariância


In [ ]:
# ============================================================
# 5. MATRIZ DE COVARIÂNCIA
# ============================================================

Cmat = collocation.covariance_matrix(
    lon_obs,
    lat_obs,
    model="gaussian",
    sigma2=sigma2,
    correlation_length=0.8,
    noise=0.05
)

print("Shape Cmat:", Cmat.shape)
print("Condicionamento aproximado:", np.linalg.cond(Cmat))

plt.figure(figsize=(6, 5))
plt.imshow(Cmat)
plt.colorbar(label="Covariância")
plt.title("Matriz de covariância gaussiana")
plt.xlabel("Observação")
plt.ylabel("Observação")
plt.show()


## 6. Predição por colocação em grade


In [ ]:
# ============================================================
# 6. PREDIÇÃO POR COLOCAÇÃO
# ============================================================

x_pred = LON.ravel()
y_pred = LAT.ravel()

pred = collocation.least_squares_collocation(
    lon_obs,
    lat_obs,
    values_obs,
    x_pred,
    y_pred,
    sigma2=sigma2,
    correlation_length=0.8,
    noise=0.3,
    model="gaussian"
)

FIELD_PRED = pred.reshape(LON.shape)

residual_grid = FIELD_TRUE - FIELD_PRED

print("Campo predito min/max:", np.nanmin(FIELD_PRED), np.nanmax(FIELD_PRED))
print("RMSE:", np.sqrt(np.nanmean(residual_grid**2)))
print("MAE:", np.nanmean(np.abs(residual_grid)))


In [ ]:
# Plot do campo verdadeiro, campo predito e resíduo

plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
c1 = plt.contourf(LON, LAT, FIELD_TRUE, levels=30, cmap="coolwarm")
plt.colorbar(c1, label="Campo")
plt.scatter(lon_obs, lat_obs, s=10, c="k")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Campo verdadeiro")

plt.subplot(1, 3, 2)
c2 = plt.contourf(LON, LAT, FIELD_PRED, levels=30, cmap="coolwarm")
plt.colorbar(c2, label="Campo")
plt.scatter(lon_obs, lat_obs, s=10, c="k")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Campo predito por colocação")

plt.subplot(1, 3, 3)
vmax = np.nanmax(np.abs(residual_grid))
c3 = plt.contourf(LON, LAT, residual_grid, levels=np.linspace(-vmax, vmax, 31), cmap="coolwarm")
plt.colorbar(c3, label="Resíduo")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Verdadeiro - Predito")

plt.tight_layout()
plt.show()


## 7. Variância de erro da predição


In [ ]:
# ============================================================
# 7. VARIÂNCIA DE ERRO
# ============================================================

err_var = collocation.collocation_error_variance(
    lon_obs,
    lat_obs,
    x_pred,
    y_pred,
    sigma2=sigma2,
    correlation_length=0.8,
    noise=0.3,
    model="gaussian"
)

ERR_STD = np.sqrt(np.maximum(err_var.reshape(LON.shape), 0.0))

print("Erro padrão min/max:", np.nanmin(ERR_STD), np.nanmax(ERR_STD))

plt.figure(figsize=(8, 5))
c = plt.contourf(LON, LAT, ERR_STD, levels=30)
plt.colorbar(c, label="Erro padrão")
plt.scatter(lon_obs, lat_obs, s=10, c="k")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Erro padrão da predição por colocação")
plt.grid(alpha=0.2)
plt.show()


## 8. Covariância empírica 1D


In [ ]:
# ============================================================
# 8. COVARIÂNCIA EMPÍRICA 1D
# ============================================================

profile = FIELD_TRUE[FIELD_TRUE.shape[0]//2, :]

cov_emp = collocation.empirical_covariance(profile, max_lag=30)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(cov_emp)), cov_emp, marker="o")
plt.xlabel("Lag")
plt.ylabel("Covariância empírica")
plt.title("Covariância empírica de um perfil do campo")
plt.grid(True)
plt.show()


## 9. Comparação entre modelos gaussiano e exponencial


In [ ]:
# ============================================================
# 9. COMPARAÇÃO GAUSSIANO VS EXPONENCIAL
# ============================================================

pred_exp = collocation.least_squares_collocation(
    lon_obs,
    lat_obs,
    values_obs,
    x_pred,
    y_pred,
    sigma2=sigma2,
    correlation_length=0.8,
    noise=0.3,
    model="exponential"
)

FIELD_PRED_EXP = pred_exp.reshape(LON.shape)

res_gauss = FIELD_TRUE - FIELD_PRED
res_exp = FIELD_TRUE - FIELD_PRED_EXP

rmse_gauss = np.sqrt(np.nanmean(res_gauss**2))
rmse_exp = np.sqrt(np.nanmean(res_exp**2))

print("RMSE Gaussiano:", rmse_gauss)
print("RMSE Exponencial:", rmse_exp)

plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
c1 = plt.contourf(LON, LAT, FIELD_TRUE, levels=30, cmap="coolwarm")
plt.colorbar(c1, label="Campo")
plt.title("Verdadeiro")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.subplot(1, 3, 2)
c2 = plt.contourf(LON, LAT, FIELD_PRED, levels=30, cmap="coolwarm")
plt.colorbar(c2, label="Campo")
plt.title("Predição Gaussiana")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.subplot(1, 3, 3)
c3 = plt.contourf(LON, LAT, FIELD_PRED_EXP, levels=30, cmap="coolwarm")
plt.colorbar(c3, label="Campo")
plt.title("Predição Exponencial")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.tight_layout()
plt.show()


## 10. Mapa global didático com Mollweide


In [ ]:
# ============================================================
# 10. MAPA GLOBAL DIDÁTICO
# ============================================================

import cartopy.crs as ccrs
import cartopy.feature as cfeature

lon_g = np.arange(-180.0, 180.0 + 2.0, 2.0)
lat_g = np.arange(-90.0, 90.0 + 2.0, 2.0)

LON_G, LAT_G = np.meshgrid(lon_g, lat_g)

FIELD_G = (
    10*np.sin(np.deg2rad(2*LON_G))*np.cos(np.deg2rad(LAT_G))
    + 5*np.exp(-((LON_G + 60)**2 + (LAT_G - 10)**2)/700)
)

fig = plt.figure(figsize=(13, 6))
ax = plt.axes(projection=ccrs.Mollweide())

cf = ax.contourf(
    LON_G,
    LAT_G,
    FIELD_G,
    levels=40,
    transform=ccrs.PlateCarree(),
    cmap="coolwarm"
)

ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.set_global()

plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.06, label="Campo sintético")
ax.set_title("Campo sintético global para contexto da colocação — Mollweide")
plt.show()


## 11. Resumo final


Se todas as células foram executadas sem erro, o módulo `collocation.py` está funcionando corretamente.

O próximo notebook testa:

```python
geodesy/gnss.py
```
